# Phase 2: Tool-Calling Trajectory Generation

Generate training data by running the **GPT-5.4 teacher agent** on ~200 diverse tickers.

**Pipeline:**
1. Fire off all tickers concurrently via `tqdm_asyncio.gather` (with semaphore for concurrency control)
2. Each task retries up to 5x with exponential backoff — failures don't crash the batch
3. Save **raw trajectories** to `data/bbb/trajectories_raw.jsonl`
4. Convert to **Hermes/chat format** for SFT (truncate tool outputs, add `<think>` tags)
5. Quality filter and save to `data/bbb/trajectories_sft.jsonl`

## Setup

In [1]:
import asyncio
import json
import os
import random
import sys
from pathlib import Path

from openai import AsyncOpenAI
from dotenv import load_dotenv
from tqdm.asyncio import tqdm_asyncio
from tenacity import retry, stop_after_attempt, wait_exponential

# Add nb/ to path so we can import bbb as a package
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "nb"))

from bbb.tools import TOOL_SCHEMAS, TOOL_FUNCTIONS
from bbb.agent import run_tool_calling_agent
from bbb.helpers__data_gen import (
    SYSTEM_PROMPT,
    TICKERS,
    FOCUS_AREAS,
    make_user_prompt,
    serialize_input_list,
    responses_to_hermes,
    filter_trajectory,
    trajectory_token_stats,
    count_tokens,
)

load_dotenv(PROJECT_ROOT / ".env")

client = AsyncOpenAI(base_url="https://us.api.openai.com/v1")

MODEL = "gpt-5.4"

# Output paths
DATA_DIR = PROJECT_ROOT / "data" / "bbb"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = DATA_DIR / "trajectories_raw.jsonl"
SFT_PATH = DATA_DIR / "trajectories_sft.jsonl"

print(f"Tickers: {len(TICKERS)}")
print(f"Raw output: {RAW_PATH}")
print(f"SFT output: {SFT_PATH}")

Tickers: 191
Raw output: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_raw.jsonl
SFT output: /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_sft.jsonl


## Configuration

- `MAX_TICKERS`: set to a small number for testing, `None` for the full run
- `MAX_CONCURRENT`: semaphore limit — controls how many agent loops run in parallel
- Retries: 5 attempts with exponential backoff (4s → 8s → 16s → 32s → 60s)

In [3]:
# Set to a small number (e.g. 5) for testing, None for the full run
MAX_TICKERS = 10

# Max concurrent agent tasks (controls yfinance + API pressure)
MAX_CONCURRENT = 20

# Teacher model reasoning effort
REASONING_EFFORT = "medium"

# Shuffle tickers for diversity in partial runs
tickers = list(TICKERS)
random.shuffle(tickers)
if MAX_TICKERS is not None:
    tickers = tickers[:MAX_TICKERS]

print(f"Will generate trajectories for {len(tickers)} tickers (max {MAX_CONCURRENT} concurrent)")
print(f"Sample: {tickers[:10]}")

Will generate trajectories for 10 tickers (max 20 concurrent)
Sample: ['ELV', 'FANG', 'ED', 'SNOW', 'ABBV', 'DAL', 'DKNG', 'MELI', 'LMT', 'ABT']


## Generate Raw Trajectories

All tickers are dispatched concurrently (bounded by semaphore). Each task:
1. Runs the teacher agent (async OpenAI calls, yfinance in thread pool)
2. Retries up to 5x on failure with exponential backoff
3. On final failure, returns an error dict — does NOT crash the batch

In [4]:
# Skip already-completed tickers (for resuming interrupted runs)
completed_tickers = set()
if RAW_PATH.exists():
    with open(RAW_PATH) as f:
        for line in f:
            rec = json.loads(line)
            completed_tickers.add(rec["ticker"])
    print(f"Resuming — {len(completed_tickers)} tickers already completed")

remaining = [t for t in tickers if t not in completed_tickers]
print(f"Remaining: {len(remaining)} tickers")

Remaining: 10 tickers


In [5]:
semaphore = asyncio.Semaphore(MAX_CONCURRENT)
# File lock for thread-safe appends
_write_lock = asyncio.Lock()


@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=4, max=60),
    reraise=True,
)
async def _run_with_retry(ticker: str, focus: str) -> dict:
    """Run the teacher agent with retries. Raises on failure."""
    async with semaphore:
        return await run_tool_calling_agent(
            client=client,
            model=MODEL,
            user_prompt=make_user_prompt(ticker, focus),
            system_prompt=SYSTEM_PROMPT,
            reasoning_effort=REASONING_EFFORT,
            verbose=False,
        )


async def generate_one(ticker: str) -> dict:
    """Generate a single trajectory. Returns error dict on final failure."""
    focus = random.choice(FOCUS_AREAS)
    try:
        result = await _run_with_retry(ticker, focus)

        record = {
            "ticker": ticker,
            "focus": focus,
            "input": serialize_input_list(result["input"]),
            "output": serialize_input_list(result["output"]),
            "tools": TOOL_SCHEMAS,
            "reasoning_summaries": result["reasoning_summaries"],
            "usage": result["usage"],
        }

        # Append to file (async lock for safety)
        async with _write_lock:
            with open(RAW_PATH, "a") as f:
                f.write(json.dumps(record) + "\n")

        return record

    except Exception as e:
        return {"ticker": ticker, "error": str(e)}

In [6]:
tasks = [generate_one(t) for t in remaining]
results = await tqdm_asyncio.gather(*tasks, desc="Generating trajectories")

# Summarize
successes = [r for r in results if "error" not in r]
errors = [r for r in results if "error" in r]

print(f"\nDone: {len(successes)} succeeded, {len(errors)} failed")
if errors:
    for e in errors:
        print(f"  {e['ticker']}: {e['error'][:100]}")

Generating trajectories: 100%|██████████| 10/10 [02:43<00:00, 16.32s/it]


Done: 10 succeeded, 0 failed


## Inspect Raw Trajectories

In [7]:
# Load all raw trajectories
raw_records = []
with open(RAW_PATH) as f:
    for line in f:
        raw_records.append(json.loads(line))

print(f"Total raw trajectories: {len(raw_records)}")

# Stats
tool_counts = []
tool_output_lengths = []
memo_lengths = []

for rec in raw_records:
    all_items = rec["input"] + rec["output"]

    calls = [it for it in all_items if isinstance(it, dict) and it.get("type") == "function_call"]
    tool_counts.append(len(calls))

    for it in all_items:
        if isinstance(it, dict) and it.get("type") == "function_call_output":
            tool_output_lengths.append(len(it["output"]))

    for it in reversed(all_items):
        if isinstance(it, dict) and it.get("type") == "message":
            for c in it.get("content", []):
                if isinstance(c, dict) and c.get("text"):
                    memo_lengths.append(len(c["text"]))
                    break
            break

if tool_counts:
    print(f"Tool calls per trajectory: min={min(tool_counts)}, max={max(tool_counts)}, avg={sum(tool_counts)/len(tool_counts):.1f}")
if tool_output_lengths:
    print(f"Tool output chars: min={min(tool_output_lengths)}, max={max(tool_output_lengths)}, avg={sum(tool_output_lengths)/len(tool_output_lengths):.0f}")
if memo_lengths:
    print(f"Memo chars: min={min(memo_lengths)}, max={max(memo_lengths)}, avg={sum(memo_lengths)/len(memo_lengths):.0f}")

Total raw trajectories: 10
Tool calls per trajectory: min=5, max=16, avg=8.6
Tool output chars: min=673, max=43981, avg=10651
Memo chars: min=2825, max=3305, avg=3093


## Convert to SFT Format (Hermes/Chat)

- Reasoning summaries → `<think>` tags
- Tool outputs truncated to ~600 tokens
- Flat schemas → nested Chat Completions format
- Quality filter: must have tool calls, valid JSON args, final output, tool diversity

In [ ]:
sft_records = []
filtered_out = []

for rec in raw_records:
    passes, reason = filter_trajectory(rec)
    if not passes:
        filtered_out.append((rec["ticker"], reason))
        continue

    hermes = responses_to_hermes(rec, max_tool_tokens=250)
    if hermes is None:
        filtered_out.append((rec["ticker"], "conversion_failed"))
        continue

    hermes["ticker"] = rec["ticker"]
    hermes["focus"] = rec.get("focus", "")
    sft_records.append(hermes)

print(f"Converted: {len(sft_records)} trajectories")
print(f"Filtered out: {len(filtered_out)}")
if filtered_out:
    for t, r in filtered_out:
        print(f"  {t}: {r}")

Converted: 10 trajectories
Filtered out: 0


In [9]:
with open(SFT_PATH, "w") as f:
    for rec in sft_records:
        f.write(json.dumps(rec) + "\n")

print(f"Saved {len(sft_records)} SFT trajectories to {SFT_PATH}")

Saved 10 SFT trajectories to /Users/kristof.rabay/Documents/code/winterfest_applied_ai_pe/data/bbb/trajectories_sft.jsonl


## Inspect a Converted Trajectory

In [10]:
if sft_records:
    sample = sft_records[0]
    print(f"Ticker: {sample['ticker']} — Focus: {sample['focus']}")
    print(f"Messages: {len(sample['messages'])} | Tools: {len(sample['tools'])}\n")

    for i, msg in enumerate(sample["messages"]):
        role = msg["role"].upper()
        content = msg.get("content", "")
        tc = msg.get("tool_calls", [])

        if role == "SYSTEM":
            print(f"[{i}] {role}: {content[:80]}...")
        elif role == "USER":
            print(f"[{i}] {role}: {content}")
        elif role == "ASSISTANT" and tc:
            think = "<think>" in (content or "")
            print(f"[{i}] {role} (think={think}, {len(tc)} tool calls): {[t['function']['name'] for t in tc]}")
        elif role == "ASSISTANT":
            print(f"[{i}] {role} (final): {len(content or '')} chars")
        elif role == "TOOL":
            print(f"[{i}] {role}: {len(content or '')} chars")
        print()

Ticker: DAL — Focus: risk factors and key challenges ahead
Messages: 9 | Tools: 4

[0] SYSTEM: You are a sell-side equity research analyst producing a brief research snapshot....

[1] USER: Research DAL focusing on risk factors and key challenges ahead.

[2] ASSISTANT (think=True, 5 tool calls): ['get_stock_news', 'get_price_history', 'get_recommendations', 'get_financials', 'get_financials']

[3] TOOL: 1912 chars

[4] TOOL: 1421 chars

[5] TOOL: 1739 chars

[6] TOOL: 1973 chars

[7] TOOL: 1963 chars

[8] ASSISTANT (final): 7451 chars



## Dataset Statistics

Token counts per role — verifies data fits within `max_seq_length=8192` for SFT.

In [11]:
if sft_records:
    all_stats = [trajectory_token_stats(rec) for rec in sft_records]

    for role in ["system", "user", "assistant", "tool", "total"]:
        vals = [s[role] for s in all_stats]
        print(f"{role:>10}: avg={sum(vals)/len(vals):.0f}  min={min(vals)}  max={max(vals)}")

    over_limit = sum(1 for s in all_stats if s["total"] > 8192)
    print(f"\nTrajectories > 8192 tokens: {over_limit}/{len(all_stats)}")

    system: avg=293  min=293  max=293
      user: avg=12  min=11  max=13
 assistant: avg=2974  min=2139  max=4469
      tool: avg=5108  min=2875  max=9522
     total: avg=8386  min=5363  max=14297

Trajectories > 8192 tokens: 3/10
